In [9]:
import boto3
import os
import logging
from dotenv import load_dotenv

In [2]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [25]:
root_path = 'C:/Users/KIIT/Desktop/Stratlytics/02_Bootcamp/04_Python/'
dealer_downloaded_file_path = root_path + '03_Output/10_dealer_downloaded.csv'
clean_dealer_file_path = root_path + '01_Data/clean/dealer.csv'

In [10]:
load_dotenv()

s3 = boto3.client(
    "s3",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name='ap-south-1'
)

logger.info("S3 client initialized successfully")

INFO:__main__:S3 client initialized successfully


In [21]:
def list_bucket_objects(bucket_name):
    '''List all objects in specified S3 bucket'''
    try:
        response = s3.list_objects_v2(Bucket=bucket_name)
        
        if "Contents" not in response:
            logger.warning(f"No objects found in {bucket_name}")
            return []
        
        for object in response.get("Contents",[]):
            logger.info(f"Found: {object['Key']}")
            
        return response["Contents"]
    except Exception as e:
        logger.error(f"Failed to list objects:{e}")
        raise
    
list_bucket_objects("raw-bucket-427763921511-ap-south-1-an")

INFO:__main__:Found: dirty/dealer.csv
INFO:__main__:Found: dirty/inventory.csv
INFO:__main__:Found: dirty/product.csv
INFO:__main__:Found: dirty/sales_logs.jsonl
INFO:__main__:Found: dirty/test_inventory.csv


[{'Key': 'dirty/dealer.csv',
  'LastModified': datetime.datetime(2026, 7, 16, 16, 44, 19, tzinfo=tzutc()),
  'ETag': '"a70a92819610e8513f45c4a516294236"',
  'ChecksumAlgorithm': ['CRC64NVME'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 9571,
  'StorageClass': 'STANDARD'},
 {'Key': 'dirty/inventory.csv',
  'LastModified': datetime.datetime(2026, 7, 16, 16, 44, 20, tzinfo=tzutc()),
  'ETag': '"a4abb419b4b31467667d48ec76e62b50"',
  'ChecksumAlgorithm': ['CRC64NVME'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 61273,
  'StorageClass': 'STANDARD'},
 {'Key': 'dirty/product.csv',
  'LastModified': datetime.datetime(2026, 7, 16, 16, 44, 20, tzinfo=tzutc()),
  'ETag': '"de638b91b62a0128274675e0b79685ee"',
  'ChecksumAlgorithm': ['CRC64NVME'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 36155,
  'StorageClass': 'STANDARD'},
 {'Key': 'dirty/sales_logs.jsonl',
  'LastModified': datetime.datetime(2026, 7, 16, 16, 44, 21, tzinfo=tzutc()),
  'ETag': '"a0e45dc9b349d06bd6958ecb52e73a89"',
  'ChecksumAlg

In [24]:
def download_from_s3(bucket,s3_key,local_path):
    '''Download file from S3 to local filesystem'''
    try:
        logger.info(f"Downloading {s3_key} from {bucket}")
        
        s3.download_file(
            bucket, s3_key, local_path
        )
        
        logger.info(f"Saved to {local_path}")
        return True
    except Exception as e: 
        logger.error(f"Download failed: {e}")
        
download_from_s3(
    "raw-bucket-427763921511-ap-south-1-an", "dirty/dealer.csv", dealer_downloaded_file_path
)

INFO:__main__:Downloading dirty/dealer.csv from raw-bucket-427763921511-ap-south-1-an
INFO:__main__:Saved to C:/Users/KIIT/Desktop/Stratlytics/02_Bootcamp/04_Python/03_Output/10_dealer_downloaded.csv


True

In [26]:
def upload_to_s3(local_path,bucket,s3_key):
    '''Upload local file to s3 bucket'''
    
    try:
        logger.info(f"Uploading {local_path} to s3://{bucket}/{s3_key}")
        
        s3.upload_file(
            local_path, bucket, s3_key
        )
        
        logger.info(f"Upload successful: {s3_key}")
        return True
    except Exception as e:
        logger.error(f"Upload failed: {e}")
        return False
    
upload_to_s3(
    clean_dealer_file_path,
    "raw-bucket-427763921511-ap-south-1-an",
    "processed/dealer_clean.csv"
)

INFO:__main__:Uploading C:/Users/KIIT/Desktop/Stratlytics/02_Bootcamp/04_Python/01_Data/clean/dealer.csv to s3://raw-bucket-427763921511-ap-south-1-an/processed/dealer_clean.csv
ERROR:__main__:Upload failed: Failed to upload C:/Users/KIIT/Desktop/Stratlytics/02_Bootcamp/04_Python/01_Data/clean/dealer.csv to raw-bucket-427763921511-ap-south-1-an/processed/dealer_clean.csv: An error occurred (AccessDenied) when calling the PutObject operation: User: arn:aws:iam::427763921511:user/BootcampS3 is not authorized to perform: s3:PutObject on resource: "arn:aws:s3:::raw-bucket-427763921511-ap-south-1-an/processed/dealer_clean.csv" because no identity-based policy allows the s3:PutObject action


False